# Improving earthquake metadata for GNSS time series analysis using Okada-based co-seismic deformation modelling
# ***Finite Source Model***
*** 
***

## **0. Workflow**

This notebook uses the Finite Source section of the Okada 1985 model [1] for surface deformations resulting from shear and tensile internal dislocations. The workflow is as follows:

1. Load all Nevada Geodetic Laboratory (NGL) Stations in Greece, previously retrieved in the point-source notebook.
2. From the Global CMT [2,3] solution from 1976 to 2020, load the data of the 10 largest earthquakes in Greece, previously retrieved in the point-source notebook.
3. Write the Okada 1985 [1] finite-source deformation code and verify it with the calibration table that Okada provided.
4. Refine the code to apply to real-life earthquakes from the Global CMT solution.
5. Apply the Scaling Laws [4-6] that IGS uses to convert a CMT point-source to a finite-source.
6. Compute the surface deformations at all relevant NGL stations from the 10 largest earthquakes in Greece.

*** 

[1] Y. Okada, “Surface deformation due to shear and tensile faults in a half-space,” Bulletin of the Seismological Society of America, vol. 75, no. 4, pp. 1135–1154, Aug. 1985. doi:10.1785/bssa0750041135

[2] A. M. Dziewonski, T.-A. Chou, and J. H. Woodhouse, “Determination of earthquake source parameters from waveform data for studies of global and regional seismicity,” Journal of Geophysical Research, vol. 86, pp. 2825–2852, 1981, doi: 10.1029/JB086iB04p02825.

[3] G. Ekström, M. Nettles, and A. M. Dziewonski, “The global CMT project 2004–2010: Centroid-moment tensors for 13,017 earthquakes,” Physics of the Earth and Planetary Interiors, vols. 200–201, pp. 1–9, 2012, doi: 10.1016/j.pepi.2012.04.002.

[4] L. Métivier, X. Collilieux, D. Lercier, Z. Altamimi, and F. Beauducel, “Global coseismic deformations, GNSS time series analysis, and Earthquake Scaling Laws,” Journal of Geophysical Research: Solid Earth, vol. 119, no. 12, pp. 9095–9109, Dec. 2014. doi:10.1002/2014jb011280 

[5] Y.-T. Yen and K.-F. Ma, “Source-scaling relationship for M 4.6-8.9 earthquakes, specifically for earthquakes in the collision zone of Taiwan,” Bulletin of the Seismological Society of America, vol. 101, no. 2, pp. 464–481, Mar. 2011. doi:10.1785/0120100046 

[6] P. M. Mai and G. C. Beroza, “Source scaling properties from finite-fault-rupture models,” Bulletin of the Seismological Society of America, vol. 90, no. 3, pp. 604–615, Jun. 2000. doi:10.1785/0119990126 

## **1. Load all Nevada Geodetic Laboratory (NGL) Stations in Greece.**

In [3]:
from obspy import read_events
from obspy import UTCDateTime
import pandas as pd

In [4]:
stations = pd.read_csv("NGL_Greece_Stations.csv", sep=",")
stations_ids = stations["Station_ID"].astype(str)
stations_lats = pd.to_numeric(stations["Latitude"], errors="coerce")
stations_lons = pd.to_numeric(stations["Longitude"], errors="coerce")
print(len(stations))

270


## **2. From the Global CMT solution from 1976 to 2026, retrieve the data of the 10 largest earthquakes in Greece.**

In [5]:
big10_catalog = read_events("Big10_Greece_Seismicity.xml")
print(len(big10_catalog))

10


## **3. Write the Okada 1985 finite-source deformation code and verify it with the calibration table that Okada provided.**

In [6]:
import numpy as np

The displacement field $u_i(x_1,x_2,x_3)$ due to a dislocation $\Delta u_j(\xi_1.\xi_2,\xi_3)$ across an internal surface $\Sigma$ in an isotropic medium is given by:

\begin{equation}
u_i=\frac{1}{F}\int\int_\Sigma\Delta u_j\biggr[\lambda\delta_{jk}\frac{\partial u_i^n}{\partial\xi_n}+\mu\biggr(\frac{\partial u_i^j}{\partial\xi_k}+\frac{\partial u_i^k}{\partial\xi_j}\biggr)\biggr]\nu_kd\Sigma \tag{1}
\end{equation}

where $\delta_{jk}$ is the Kronecker delta, $\lambda$ and $\mu$ are Lame parameters, $\nu_k$ is the direction cosine of the normal to the surface element $d\Sigma$, and the summation convention applies. $u_i^j$ is the $i$th component of the displacement at $(x_1,x_2,x_3)$ due to the $j$th direction point force of magnitude $F$ at $(\xi_1,\xi_2\xi_3)$ [7].

Okada [1] provides a compact solution for this.

***

[7] J. A. Steketee, “Some geophysical applications of the elasticity theory of dislocations,” Can. J. Phys., vol. 36, no. 9, pp. 1168–1197, Sep. 1958. [Online]. Available: Canadian Science Publishing

### 3.1 Translation into Okada's Coordinate System

These intermediate varaibles are important for the translation into Okada's finite source coordinate system.

\begin{cases}
p=y\cos\delta+d\sin\delta\\
q=y\sin\delta-d\cos\delta\\
\tilde{y}=\eta\cos\delta+q\sin\delta\\
\tilde{d}=\eta\sin\delta-q\cos\delta\\
R^2=\xi^2+\eta^2+q^2=\xi^2+\tilde{y}^2+\tilde{d}^2\\
X^2=\xi^2+q^2
\tag{2}
\end{cases}

### 3.2 Chinnery Notation

To apply the Chinnery notation [8] to Table 2 of Okada 1985, we must follow the given formula.

\begin{equation} f(\xi, \eta) \parallel = f(x, p) - f(x, p-W) - f(x-L, p) + f(x-L, p-W) \tag{3} \end{equation}

***

[8] M. A. Chinnery, “The deformation of the ground around surface faults,” Bulletin of the Seismological Society of America, vol. 51, no. 3, pp. 355–372, Jul. 1961. doi:10.1785/bssa0510030355 

In [7]:
def coords_sim (stalon, stalat, evlon, evlat, evdep, strike, dip, rake, mu, lam, opening, U):
    dEast = stalon - evlon #m
    dNorth = stalat - evlat #m

    x = dEast * np.sin(strike*np.pi/180.0) + dNorth * np.cos(strike*np.pi/180.0) #m
    y = -dEast * np.cos(strike*np.pi/180.0) + dNorth * np.sin(strike*np.pi/180.0) #m
    d = abs(evdep) #m
    delta = dip*np.pi/180.0 #radians
    
    p = y*np.cos(delta)+d*np.sin(delta) #m
    q = y*np.sin(delta)-d*np.cos(delta) #m

    mu = 10**9*mu #convert GPa to N/m^2
    lam = 10**9*lam #convert GPa to N/m^2
    return(
        U*np.cos(rake*np.pi/180.0), #m #U1
        U*np.sin(rake*np.pi/180.0), #m #U2
        opening, #m #U3
        x, #m #x
        y, #m #y
        d, #m #d
        mu, #GPa->N/m^2 #mu
        lam, #GPa->N/m^2 #lam
        delta, #rad #delta
        p, #m #p
        q, #m #q
        delta #radians #delta
    )

def chinnery_sim (x, p, q, L, W, delta):
        xi = [x,x-L] #m #xi
        eta = [p,p-W] #m #eta

        ytil = [0,0]
        dtil = [0,0]
        R = [[0,0],[0,0]]
        X = [0,0]

        for i in range(2):
            for j in range(2):
                ytil[j]=eta[j]*np.cos(delta)+q*np.sin(delta) #m #ytil
                dtil[j]=eta[j]*np.sin(delta)-q*np.cos(delta) #m #dtil
                R[i][j]=np.sqrt(xi[i]**2.0+eta[j]**2.0+q**2.0) #m #R
                X[i]=np.sqrt(xi[i]**2+q**2) #m #X

        return(xi, eta, ytil, dtil, R, X)

### 3.3 Displacements

For Strike-Slip:
\begin{cases}
u_x=-\frac{U_1}{2\pi}\biggr[\frac{\xi q}{R(R+\eta)}+\tan^{-1}\frac{\xi\eta}{qR}+I_1\sin\delta\biggr]\parallel\\
u_y=-\frac{U_1}{2\pi}\biggr[\frac{\tilde{y}q}{R(R+\eta)}+\frac{q\cos\delta}{R+\eta}+I_2\sin\delta\biggr]\parallel\\
u_z=-\frac{U_1}{2\pi}\biggr[\frac{\tilde{d}q}{R(R+\eta)}+\frac{q\sin\delta}{R+\eta}+I_4\sin\delta\biggr]\parallel
\tag{4}
\end{cases}


For Dip-Slip:
\begin{cases}
u_x=-\frac{U_2}{2\pi}\biggr[\frac{q}{R}-I_3\sin\delta\cos\delta\biggr]\parallel\\
u_y=-\frac{U_2}{2\pi}\biggr[\frac{\tilde{y}q}{R(R+\xi)}+\cos\delta\tan^{-1}\frac{\xi\eta}{qR}-I_1\sin\delta\cos\delta\biggr]\parallel\\
u_z=-\frac{U_2}{2\pi}\biggr[\frac{\tilde{d}q}{R(R+\xi)}+\sin\delta\tan^{-1}\frac{\xi\eta}{qR}-I_5\sin\delta\cos\delta\biggr]\parallel
\tag{5}
\end{cases}

For Tensile Fault:
\begin{cases}
u_x=\frac{U_3}{2\pi}\biggr[\frac{q^2}{R(R+\eta)}-I_3\sin^2\delta\biggr]\parallel\\
u_y=\frac{U_3}{2\pi}\biggr[\frac{-\tilde{d}q}{R(R+\xi)}-\sin\delta\biggr(\frac{\xi q}{R(R+\eta)}-\tan^{-1}\frac{\xi\eta}{qR}\biggr)-I_1\sin^2\delta\biggr]\parallel\\
u_z=\frac{U_3}{2\pi}\biggr[\frac{\tilde{y}q}{R(R+\xi)}+\cos\delta\biggr(\frac{\xi q}{R(R+\eta)}-\tan^{-1}\frac{\xi\eta}{qR}\biggr)-I_5\sin^2\delta\biggr]\parallel\\
\tag{6}
\end{cases}

For equations $(4)$ to $(6)$, in the special case that $q=0$, set $\tan^{-1}(\xi\eta/qR)=0$ to prevent singularities.

For all equations, when $R+\eta=0$ (which occurs when $\sin\delta<0$ and $\xi=q=0$), set the terms with $R+\eta$ in their denominator to $0$ to prevent singularities.

In [8]:
def disp_strike_sim (xi, eta, q, ytil, dtil, R, I1, I2, I4, delta, U1):
    if(q==0):
        atanterm=0
    else:
        atanterm=np.atan((xi*eta)/(q*R))
    if ((R+eta)==0):
        Retaterm=0
    else:
        Retaterm=1/(R+eta)
    return (
    -U1/(2.0*np.pi)*(((xi*q)/R*Retaterm)+atanterm+I1*np.sin(delta)),
    -U1/(2.0*np.pi)*(((ytil*q)/R*Retaterm)+(q*np.cos(delta))*Retaterm+I2*np.sin(delta)),
    -U1/(2.0*np.pi)*(((dtil*q)/R*Retaterm)+(q*np.sin(delta))*Retaterm+I4*np.sin(delta))
    )

def disp_dip_sim (xi, eta, q, ytil, dtil, R, I1, I3, I5, delta, U2):
    if(q==0):
        atanterm=0
    else:
        atanterm=np.atan((xi*eta)/(q*R))
    return (
    -U2/(2.0*np.pi)*(((q/R)-I3*np.sin(delta)*np.cos(delta))),
    -U2/(2.0*np.pi)*(((ytil*q)/(R*(R+xi))+np.cos(delta)*atanterm-I1*np.sin(delta)*np.cos(delta))),
    -U2/(2.0*np.pi)*(((dtil*q)/(R*(R+xi))+np.sin(delta)*atanterm-I5*np.sin(delta)*np.cos(delta)))
    )

def disp_tens_sim (xi, eta, q, ytil, dtil, R, I1, I3, I5, delta, U3):
    if(q==0):
        atanterm=0
    else:
        atanterm=np.atan((xi*eta)/(q*R))
    if ((R+eta)==0):
        Retaterm=0
    else:
        Retaterm=1/(R+eta)
    return (
    U3/(2.0*np.pi)*(((q**2.0)/(R)*Retaterm)-I3*(np.sin(delta))**2.0),
    U3/(2.0*np.pi)*((-dtil*q)/(R*(R+xi))-np.sin(delta)*(xi*q/R*Retaterm-atanterm)-I1*np.sin(delta)**2.0),
    U3/(2.0*np.pi)*((ytil*q)/(R*(R+xi))+np.cos(delta)*(xi*q/R*Retaterm-atanterm)-I5*np.sin(delta)**2.0)
    )

Wherein the intermediate variables are:

\begin{cases}
I_1=\frac{\mu}{\lambda+\mu}\biggr[\frac{-1}{\cos\delta}\frac{\xi}{R+\tilde{d}}\biggr]-\frac{\sin\delta}{\cos\delta}I_5\\
I_2=\frac{\mu}{\lambda+\mu}\biggr[-\ln(R+\eta)\biggr]-I_3\\
I_3=\frac{\mu}{\lambda+\mu}\biggr[\frac{1}{\cos\delta}\frac{\tilde{y}}{R+\tilde{d}}-\ln(R+\eta)\biggr]+\frac{\sin\delta}{\cos\delta}I_4\\
I_4=\frac{\mu}{\lambda+\mu}\frac{1}{\cos\delta}\biggr[\ln(R+\tilde{d})-\sin\delta\ln(R+\eta)\biggr]\\
I_5=\frac{\mu}{\lambda+\mu}\frac{2}{\cos\delta}\tan^{-1}\biggr[\frac{\eta(X+q\cos\delta)+X(R+X)\sin\delta}{\xi(R+X)\cos\delta}\biggr]
\tag{7}
\end{cases}

For equation $(7)$, for the special case that $\xi=0$, set $I_5=0$ to prevent singularities.

For $\cos\delta=0$:
\begin{cases}
I_1=-\frac{\mu}{2(\lambda+\mu)}\frac{\xi q}{(R+\tilde{d})^2}\\
I_3=\frac{\mu}{2(\lambda+\mu)}\biggr[\frac{\eta}{R+\tilde{d}}+\frac{\tilde{y}q}{(R+\tilde{d})^2}-\ln(R+\eta)\biggr]\\
I_4=-\frac{\mu}{\lambda+\mu}\frac{q}{R+\tilde{d}}\\
I_5=-\frac{\mu}{\lambda+\mu}\frac{\xi\sin\delta}{R+\tilde{d}}
\tag{8}
\end{cases}

For equations $(7)$ and $(8)$, when $R+\eta=0$ (which occurs when $\sin\delta<0$ and $\xi=q=0$), replace $\ln(R+\eta)$ to $-\ln(R-\eta)$ to prevent singularities.

In [9]:
def disp_I_sim (xi, eta, q, ytil, dtil, R, X, lam, mu, delta):
    if ((R+eta)==0):
        lnterm = -np.log(R-eta)
    else:
        lnterm = np.log(R+eta)
    if (np.cos(delta)==0):
        I3 = mu/(2*(lam+mu))*(eta/(R+dtil)+(ytil*q)/(R+dtil)**2-lnterm)
        I2_1stterm =mu/(lam+mu)*(-lnterm)-I3
        return(
            -mu/(2*(lam+mu))*xi*q/(R+dtil)**2,
            I2_1stterm,
            I3,
            mu/(lam+mu)*q/(R+dtil),
            mu/(lam+mu)*xi*np.sin(delta)/(R+dtil)
        )
    else:
        I4 = mu/(lam+mu)/np.cos(delta)*(np.log(R+dtil)-np.sin(delta)*lnterm)
        I3 = mu/(lam+mu)*(1/np.cos(delta)*ytil/(R+dtil)-lnterm)+np.sin(delta)/np.cos(delta)*I4
        I5 = 0 if xi==0 else mu/(lam+mu)*2.0/np.cos(delta)*np.atan((eta*(X+q*np.cos(delta))+X*(R+X)*np.sin(delta))/(xi*(R+X)*np.cos(delta)))
        I2_1stterm =mu/(lam+mu)*(-lnterm)-I3
        return(
            mu/(lam+mu)*(-1)/np.cos(delta)*xi/(R+dtil)-np.sin(delta)/(np.cos(delta))*I5,
            I2_1stterm,
            I3,
            I4, 
            I5
        )

Putting it all together:

In [10]:
def displacement_sim (stalon, stalat, evlon, evlat, evdep, strike, dip, rake, mu, lam, opening, U, L, W) :
    U1, U2, U3, x, y, d, mu, lam, delta, p, q, delta = coords_sim (stalon, stalat, evlon, evlat, evdep, strike, dip, rake, mu, lam, opening, U)
    xi, eta, ytil, dtil, R, X = chinnery_sim (x, p, q, L, W, delta)
    ux, uy, uz = [], [], []
    for i in range(2):
        for j in range (2):
            I1, I2, I3, I4, I5 = disp_I_sim (xi[i], eta[j], q, ytil[j], dtil[j], R[i][j], X[i], lam, mu, delta)
            ux_strike, uy_strike, uz_strike = disp_strike_sim (xi[i], eta[j], q, ytil[j], dtil[j], R[i][j], I1, I2, I4, delta, U1)
            ux_dip, uy_dip, uz_dip = disp_dip_sim (xi[i], eta[j], q, ytil[j], dtil[j], R[i][j], I1, I3, I5, delta, U2)
            ux_tens, uy_tens, uz_tens = disp_tens_sim (xi[i], eta[j], q, ytil[j], dtil[j], R[i][j], I1, I3, I5, delta, U3)
            ux.append(ux_strike + ux_dip + ux_tens)
            uy.append(uy_strike + uy_dip + uy_tens)
            uz.append(uz_strike + uz_dip + uz_tens)
    return ( #chinnery
        ux[0]-ux[1]-ux[2]+ux[3], #ux
        uy[0]-uy[1]-uy[2]+uy[3], #uy
        uz[0]-uz[1]-uz[2]+uz[3] #uz
    )

Testing with Table 2 of Okada 1985:

Recall that the rake angles are:
- Normal Fault: -90 deg
- Reverse/ Thrust: 90 deg
- Strike Slip: 0 or 180 deg

In [11]:
stalon, stalat, stadep, evlon, evlat, evdep = -3.0, 2.0, 0.0, 0.0, 0.0, 4.0 #x=2, y=3, d=4
strike, dip = 0.0, 70.0 #degrees
mu, lam = 30, 30 #GPa
L, W = 3, 2

print("Strike-Slip Results: -8.689E-3, -4.298E-3, -2.747E-3")
rake, opening, U = 0.0, 0.0, 1.0
ux, uy, uz = displacement_sim (stalon, stalat, evlon, evlat, evdep, strike, dip, rake, mu, lam, opening, U, L, W)
print("ux={:0.3e}".format(ux))
print("uy={:0.3e}".format(uy))
print("uz={:0.3e}".format(uz))

print("- - - - - - - - - - - - - - - - - - - - - - - - - -")

print("Dip-Slip Results: -4.682E-3, -3.527E-2, -3.564E-2")
rake, opening, U = 90.0, 0.0, 1.0
ux, uy, uz = displacement_sim (stalon, stalat, evlon, evlat, evdep, strike, dip, rake, mu, lam, opening, U, L, W)
print("ux={:0.3e}".format(ux))
print("uy={:0.3e}".format(uy))
print("uz={:0.3e}".format(uz))

print("- - - - - - - - - - - - - - - - - - - - - - - - - -")

print("Tensile Results: -2.660E-4, +1.056E-2, +3.214E-3")
rake, opening, U = 0.0, 1.0, 0.0
ux, uy, uz = displacement_sim (stalon, stalat, evlon, evlat, evdep, strike, dip, rake, mu, lam, opening, U, L, W)
print("ux={:0.3e}".format(ux))
print("uy={:0.3e}".format(uy))
print("uz={:0.3e}".format(uz))

Strike-Slip Results: -8.689E-3, -4.298E-3, -2.747E-3
ux=-8.689e-03
uy=-4.298e-03
uz=-2.747e-03
- - - - - - - - - - - - - - - - - - - - - - - - - -
Dip-Slip Results: -4.682E-3, -3.527E-2, -3.564E-2
ux=-4.682e-03
uy=-3.527e-02
uz=-3.564e-02
- - - - - - - - - - - - - - - - - - - - - - - - - -
Tensile Results: -2.660E-4, +1.056E-2, +3.214E-3
ux=-2.660e-04
uy=1.056e-02
uz=3.214e-03


### 3.4 Strains

For Strike-Slip:
\begin{cases}
\frac{\partial u_x}{\partial x}=\frac{U_1}{2\pi}\biggr[\xi^2qA_\eta-J_1\sin\delta\biggr]\parallel\\
\frac{\partial u_x}{\partial y}=\frac{U_1}{2\pi}\biggr[\frac{\xi^3\tilde{d}}{R^3(\eta^2+q^2)}-(\xi^3A_\eta+J_2)\sin\delta\biggr]\parallel\\
\frac{\partial u_y}{\partial x}=\frac{U_1}{2\pi}\biggr[\frac{\xi q}{R^3}\cos\delta+(\xi q^2A_\eta-J_2)\sin\delta\biggr]\parallel\\
\frac{\partial u_y}{\partial y}=\frac{U_1}{2\pi}\biggr[\frac{\tilde{y}q}{R^3}\cos\delta+\biggr(q^3A_\eta\sin\delta-\frac{2q\sin\delta}{R(R+\eta)}-\frac{\xi^2+\eta^2}{R^3}\cos\delta-J_4\biggr)\sin\delta\biggr]\parallel
\tag{9}
\end{cases}

For Dip-Slip:
\begin{cases}
\frac{\partial u_x}{\partial x}=\frac{U_2}{2\pi}\biggr[\frac{\xi q}{R^3}+J_3\sin\delta\cos\delta\biggr]\parallel\\
\frac{\partial u_x}{\partial y}=\frac{U_2}{2\pi}\biggr[\frac{\tilde{y}q}{R^3}-\frac{\sin\delta}{R}+J_1\sin\delta\cos\delta\biggr]\parallel\\
\frac{\partial u_y}{\partial x}=\frac{U_2}{2\pi}\biggr[\frac{\tilde{y}q}{R^3}+\frac{q\cos\delta}{R(R+\eta)}+J_1\sin\delta\cos\delta\biggr]\parallel\\
\frac{\partial u_y}{\partial y}=\frac{U_2}{2\pi}\biggr[\tilde{y}^2 qA_\xi-\biggr(\frac{2\tilde{y}}{R(R+\xi)}+\frac{\xi\cos\delta}{R(R+\eta)}\biggr)\sin\delta+J_2\sin\delta\cos\delta\biggr]\parallel
\tag{10}
\end{cases}

For Tensile Fault:
\begin{cases}
\frac{\partial u_x}{\partial x}=-\frac{U_3}{2\pi}\biggr[\xi q^2A_\eta+J_3\sin^2\delta\biggr]\parallel\\
\frac{\partial u_x}{\partial y}=-\frac{U_3}{2\pi}\biggr[-\frac{\tilde{d}q}{R^3}-\xi^2qA_\eta\sin\delta+J_1\sin^2\delta\biggr]\parallel\\
\frac{\partial u_y}{\partial x}=-\frac{U_3}{2\pi}\biggr[\frac{q^2}{R^3}\cos\delta+q^3A_\eta\sin\delta+J_1\sin^2\delta\biggr]\parallel\\
\frac{\partial u_y}{\partial y}=-\frac{U_3}{2\pi}\biggr[(\tilde{y}\cos\delta-\tilde{d}\sin\delta)q^2A_\xi-\frac{q\sin2\delta}{R(R+\xi)}-(\xi q^2A_\eta-J_2)\sin^2\delta\biggr]\parallel
\tag{11}
\end{cases}

In [12]:
def strain_strike_sim (xi, eta, q, ytil, dtil, R, J1, J2, J4, delta, U1):
    Aeta=(2*R+eta)/(R**3*(R+eta)**2)
    if ((R+eta)==0):
        Retaterm=0
    else:
        Retaterm=1/(R+eta)
    return (
        U1/(2.0*np.pi)*(xi**2.0*q*Aeta-J1*np.sin(delta)),
        U1/(2.0*np.pi)*((xi**3.0*dtil)/(R**3.0*(eta**2.0+q**2.0))-(xi**3.0*Aeta+J2)*np.sin(delta)),
        U1/(2.0*np.pi)*(xi*q/R**3.0*np.cos(delta)+(xi*q**2.0*Aeta-J2)*np.sin(delta)),
        U1/(2.0*np.pi)*(ytil*q/R**3.0*np.cos(delta)+(q**3.0*Aeta*np.sin(delta)-2.0*q*np.sin(delta)/R*Retaterm-(xi**2+eta**2)/R**3*np.cos(delta)-J4)*np.sin(delta))
    )

def strain_dip_sim (xi, eta, q, ytil, dtil, R, J1, J2, J3, delta, U2):
    Axi=(2*R+xi)/(R**3*(R+xi)**2)
    if ((R+eta)==0):
        Retaterm=0
    else:
        Retaterm=1/(R+eta)
    return (
        U2/(2.0*np.pi)*(xi*q/R**3.0+J3*np.sin(delta)*np.cos(delta)),
        U2/(2.0*np.pi)*(ytil*q/R**3.0-np.sin(delta)/R+J1*np.sin(delta)*np.cos(delta)),
        U2/(2.0*np.pi)*(ytil*q/R**3.0+q*np.cos(delta)/R*Retaterm+J1*np.sin(delta)*np.cos(delta)),
        U2/(2.0*np.pi)*(ytil**2*q*Axi-(2.0*ytil/(R*(R+xi))+xi*np.cos(delta)/R*Retaterm)*np.sin(delta)+J2*np.sin(delta)*np.cos(delta))
    )

def strain_tens_sim (xi, eta, q, ytil, dtil, R, J1, J2, J3, delta, U3):
    Axi=(2*R+xi)/(R**3*(R+xi)**2)
    Aeta=(2*R+eta)/(R**3*(R+eta)**2)
    return (
        -U3/(2.0*np.pi)*(xi*q**2.0*Aeta+J3*(np.sin(delta))**2.0),
        -U3/(2.0*np.pi)*(-dtil*q/R**3.0-xi**2*q*Aeta*np.sin(delta)+J1*(np.sin(delta))**2.0),
        -U3/(2.0*np.pi)*(q**2.0/R**3.0*np.cos(delta)+q**3.0*Aeta*np.sin(delta)+J1*(np.sin(delta))**2.0),
        -U3/(2.0*np.pi)*((ytil*np.cos(delta)-dtil*np.sin(delta))*q**2.0*Axi-q*np.sin(2.0*delta)/(R*(R+xi))-(xi*q**2*Aeta-J2)*(np.sin(delta))**2.0)
    )

Wherein the intermediate values are:

\begin{cases}
J_1=\frac{\mu}{\lambda+\mu}\frac{1}{\cos\delta}\biggr[\frac{\xi^2}{R(R+\tilde{d})^2}-\frac{1}{R+\tilde{d}}\biggr]-\frac{\sin\delta}{\cos\delta}K_3\\
J_2=\frac{\mu}{\lambda+\mu}\frac{1}{\cos\delta}\biggr[\frac{\xi\tilde{y}}{R(R+\tilde{d})^2}\biggr]-\frac{\sin\delta}{\cos\delta}K_1\\
J_3=\frac{\mu}{\lambda+\mu}\biggr[-\frac{\xi}{R(R+\eta)}\biggr]-J_2\\
J_4=\frac{\mu}{\lambda+\mu}\biggr[-\frac{\cos\delta}{R}-\frac{q\sin\delta}{R(R+\eta)}\biggr]-J_1
\tag{12}
\end{cases}

$K_1$ and $K_3$ are given in equation $(17)$, and if $\cos\delta=0$:

\begin{cases}
J_1=\frac{\mu}{2(\lambda+\mu)}\frac{q}{(R+\tilde{d})^2}\biggr[\frac{2\xi^2}{R(R+\tilde{d})}-1\biggr]\\
J_2=\frac{\mu}{2(\lambda+\mu)}\frac{\xi\sin\delta}{(R+\tilde{d})^2}\biggr[\frac{2\xi^2}{R(R+\tilde{d})}-1\biggr]
\tag{13}
\end{cases}

\begin{cases}
A_\xi=\frac{2R+\xi}{R^3(R+\xi)^2}\\
A_\eta=\frac{2R+\eta}{R^3(R+\eta)^2}
\tag{14}
\end{cases}

In [13]:
def strain_J_sim (xi, eta, q, ytil, dtil, R, lam, mu, delta):
    K1, K2, K3 = tilt_K_sim (xi, eta, q, ytil, dtil, R, lam, mu, delta)
    if ((R+eta)==0):
        Retaterm=0
    else:
        Retaterm=1/(R+eta)
    if (np.cos(delta)==0):
        J1=mu/(2.0*(lam+mu))*q/(R+dtil)**2.0*(2.0*xi**2.0/(R*(R+dtil))-1.0)
        J2=mu/(2.0*(lam+mu))*xi*np.sin(delta)/(R+dtil)**2.0*(2.0*xi**2.0/(R*(R+dtil))-1)
        return(
            J1,
            J2,
            mu/(lam+mu)*(-xi/R*Retaterm)-J2,
            mu/(lam+mu)*(-np.cos(delta)/R-q*np.sin(delta)/R*Retaterm)-J1
        )
    else:
        J1=mu/(lam+mu)/np.cos(delta)*(xi**2.0/(R*(R+dtil)**2.0)-1.0/(R+dtil))-np.sin(delta)/np.cos(delta)*K3
        J2=mu/(lam+mu)/np.cos(delta)*xi*ytil/(R*(R+dtil)**2.0)-np.sin(delta)/np.cos(delta)*K1
        return(
            J1,
            J2,
            mu/(lam+mu)*(-xi/R*Retaterm)-J2,
            mu/(lam+mu)*(-np.cos(delta)/R-q*np.sin(delta)/R*Retaterm)-J1
        )

Putting it all together:

In [14]:
def strain_sim (stalon, stalat, evlon, evlat, evdep, strike, dip, rake, mu, lam, opening, U, L, W) :
    U1, U2, U3, x, y, d, mu, lam, delta, p, q, delta = coords_sim (stalon, stalat, evlon, evlat, evdep, strike, dip, rake, mu, lam, opening, U)
    xi, eta, ytil, dtil, R, X = chinnery_sim (x, p, q, L, W, delta)
    uxx, uxy, uyx, uyy = [], [], [], []
    for i in range(2):
        for j in range (2):
            J1, J2, J3, J4 = strain_J_sim (xi[i], eta[j], q, ytil[j], dtil[j], R[i][j], lam, mu, delta)
            uxx_strike, uxy_strike, uyx_strike, uyy_strike = strain_strike_sim (xi[i], eta[j], q, ytil[j], dtil[j], R[i][j], J1, J2, J4, delta, U1)
            uxx_dip, uxy_dip, uyx_dip, uyy_dip = strain_dip_sim (xi[i], eta[j], q, ytil[j], dtil[j], R[i][j], J1, J2, J3, delta, U2)
            uxx_tens, uxy_tens, uyx_tens, uyy_tens = strain_tens_sim (xi[i], eta[j], q, ytil[j], dtil[j], R[i][j], J1, J2, J3, delta, U3)
            uxx.append(uxx_strike + uxx_dip + uxx_tens)
            uxy.append(uxy_strike + uxy_dip + uxy_tens)
            uyx.append(uyx_strike + uyx_dip + uyx_tens)
            uyy.append(uyy_strike + uyy_dip + uyy_tens)
    return ( #chinnery
        uxx[0]-uxx[1]-uxx[2]+uxx[3], #uxx
        uxy[0]-uxy[1]-uxy[2]+uxy[3], #uxy
        uyx[0]-uyx[1]-uyx[2]+uyx[3], #uyx
        uyy[0]-uyy[1]-uyy[2]+uyy[3] #uyy
    )

Testing with Table 2 of Okada 1985:

In [17]:
stalon, stalat, stadep, evlon, evlat, evdep = -3.0, 2.0, 0.0, 0.0, 0.0, 4.0 #x=2, y=3, d=4
strike, dip = 0.0, 70.0 #degrees
mu, lam = 30, 30 #GPa
L, W = 3, 2

print("Strike-Slip Results: -1.220E-3, +2.470E-4, -8.191E-3, -5.814E-4")
rake, opening, U = 0.0, 0.0, 1.0
uxx, uxy, uyx, uyy = strain_sim (stalon, stalat, evlon, evlat, evdep, strike, dip, rake, mu, lam, opening, U, L, W)
print("uxx={:0.3e}".format(uxx))
print("uxy={:0.3e}".format(uxy))
print("uyx={:0.3e}".format(uyx))
print("uyy={:0.3e}".format(uyy))

print("- - - - - - - - - - - - - - - - - - - - - - - - - -")

print("Dip-Slip Results: -8.867E-3, -1.519E-4, +4.057E-3, -1.035E-2")
rake, opening, U = 90.0, 0.0, 1.0
uxx, uxy, uyx, uyy = strain_sim (stalon, stalat, evlon, evlat, evdep, strike, dip, rake, mu, lam, opening, U, L, W)
print("uxx={:0.3e}".format(uxx))
print("uxy={:0.3e}".format(uxy))
print("uyx={:0.3e}".format(uyx))
print("uyy={:0.3e}".format(uyy))

print("- - - - - - - - - - - - - - - - - - - - - - - - - -")

print("Tensile Results: -5.655E-4, +1.993E-3, -1.066E-3, +1.230E-2")
rake, opening, U = 0.0, 1.0, 0.0
uxx, uxy, uyx, uyy = strain_sim (stalon, stalat, evlon, evlat, evdep, strike, dip, rake, mu, lam, opening, U, L, W)
print("uxx={:0.3e}".format(uxx))
print("uxy={:0.3e}".format(uxy))
print("uyx={:0.3e}".format(uyx))
print("uyy={:0.3e}".format(uyy))

Strike-Slip Results: -1.220E-3, +2.470E-4, -8.191E-3, -5.814E-4
uxx=-1.220e-03
uxy=2.470e-04
uyx=-8.191e-03
uyy=-5.814e-04
- - - - - - - - - - - - - - - - - - - - - - - - - -
Dip-Slip Results: -8.867E-3, -1.519E-4, +4.057E-3, -1.035E-2
uxx=-8.867e-03
uxy=-1.519e-04
uyx=4.057e-03
uyy=-1.035e-02
- - - - - - - - - - - - - - - - - - - - - - - - - -
Tensile Results: -5.655E-4, +1.993E-3, -1.066E-3, +1.230E-2
uxx=-5.655e-04
uxy=1.993e-03
uyx=-1.066e-03
uyy=1.230e-02


### 3.5 Tilts

For Strike-Slip:
\begin{cases}
\frac{\partial u_z}{\partial x}=\frac{U_1}{2\pi}\biggr[-\xi q^2A_\eta\cos\delta+\biggr(\frac{\xi q}{R^3}-K_1\biggr)\sin\delta\biggr]\parallel\\
\frac{\partial u_z}{\partial y}=\frac{U_1}{2\pi}\biggr[\frac{\tilde{d}q}{R^3}\cos\delta+\biggr(\xi^2qA_\eta\cos\delta-\frac{\sin\delta}{R}+\frac{\tilde{y}q}{R^3}-K_2\biggr)\sin\delta\biggr]\parallel
\tag{15}
\end{cases}

For Dip-Slip:
\begin{cases}
\frac{\partial u_z}{\partial x}=\frac{U_2}{2\pi}\biggr[\frac{\tilde{d}q}{R^3}+\frac{q\sin\delta}{R(R+\eta)}+K_3\sin\delta\cos\delta\biggr]\parallel\\
\frac{\partial u_z}{\partial y}=\frac{U_2}{2\pi}\biggr[\tilde{y}\tilde{d}qA_\xi-\biggr(\frac{2\tilde{d}}{R(R+\xi)}+\frac{\xi\sin\delta}{R(R+\eta)}\biggr)\sin\delta+K_1\sin\delta\cos\delta\biggr]\parallel
\tag{16}
\end{cases}

For Tensile Fault:
\begin{cases}
\frac{\partial u_z}{\partial x}=-\frac{U_3}{2\pi}\biggr[\frac{q^2}{R^3}\sin\delta-q^3A_\eta\cos\delta+K_3\sin^2\delta\biggr]\parallel\\
\frac{\partial u_z}{\partial y}=-\frac{U_3}{2\pi}\biggr[(\tilde{y}\sin\delta+\tilde{d}\cos\delta)q^2A_\xi+\xi q^2A_\eta\sin\delta\cos\delta-\biggr(\frac{2q}{R(R+\xi)}-K_1\biggr)\sin^2\delta\biggr]\parallel
\tag{17}
\end{cases}

In [18]:
def tilt_strike_sim (xi, eta, q, ytil, dtil, R, K1, K2, delta, U1):
    Aeta=(2*R+eta)/(R**3*(R+eta)**2)
    return (
        U1/(2.0*np.pi)*(-xi*q**2.0*Aeta*np.cos(delta)+(xi*q/R**3.0-K1)*np.sin(delta)),
        U1/(2.0*np.pi)*(dtil*q/R**3*np.cos(delta)+(xi**2.0*q*Aeta*np.cos(delta)-np.sin(delta)/R+ytil*q/R**3-K2)*np.sin(delta))
    )

def tilt_dip_sim (xi, eta, q, ytil, dtil, R, K1, K3, delta, U2):
    Axi=(2*R+xi)/(R**3*(R+xi)**2)
    if ((R+eta)==0):
        Retaterm=0
    else:
        Retaterm=1/(R+eta)
    return (
        U2/(2.0*np.pi)*(dtil*q/R**3+q*np.sin(delta)/R*Retaterm+K3*np.sin(delta)*np.cos(delta)),
        U2/(2.0*np.pi)*(ytil*dtil*q*Axi-(2.0*dtil/(R*(R+xi))+xi*np.sin(delta)/R*Retaterm)*np.sin(delta)+K1*np.sin(delta)*np.cos(delta))
    )

def tilt_tens_sim (xi, eta, q, ytil, dtil, R, K1, K3, delta, U3):
    Axi=(2*R+xi)/(R**3*(R+xi)**2)
    Aeta=(2*R+eta)/(R**3*(R+eta)**2)
    return (
        -U3/(2.0*np.pi)*(q**2.0/R**3.0*np.sin(delta)-q**3.0*Aeta*np.cos(delta)+K3*(np.sin(delta))**2),
        -U3/(2.0*np.pi)*((ytil*np.sin(delta)+dtil*np.cos(delta))*q**2.0*Axi+xi*q**2*Aeta*np.sin(delta)*np.cos(delta)-(2.0*q/(R*(R+xi))-K1)*(np.sin(delta))**2)
    )

Wherein the intermediate values are:

\begin{cases}
K_1=\frac{\mu}{\lambda+\mu}\frac{\xi}{\cos\delta}\biggr[\frac{1}{R(R+\tilde{d})}-\frac{\sin\delta}{R(R+\eta)}\biggr]\\
K_2=\frac{\mu}{\lambda+\mu}\biggr[-\frac{\sin\delta}{R}+\frac{q\cos\delta}{R(R+\eta)}\biggr]-K_3\\
K_3=\frac{\mu}{\lambda+\mu}\frac{1}{\cos\delta}\biggr[\frac{q}{R(R+\eta)}-\frac{\tilde{y}}{R(R+\tilde{d})}\biggr]
\tag{18}
\end{cases}

And if $\cos\delta=0$:
\begin{cases}
K_1=\frac{\mu}{\lambda+\mu}\frac{\xi q}{R(R+\tilde{d})^2}\\
K_3=\frac{\mu}{\lambda+\mu}\frac{\sin\delta}{R+\tilde{d}}\biggr[\frac{\xi^2}{R(R+\tilde{d})}-1\biggr]
\tag{19}
\end{cases}

In [19]:
def tilt_K_sim (xi, eta, q, ytil, dtil, R, lam, mu, delta):
    if ((R+eta)==0):
        Retaterm=0
    else:
        Retaterm=1/(R+eta)
    if (np.cos(delta)==0):
        K3=mu/(lam+mu)*np.sin(delta)/(R+dtil)*(xi**2/(R*(R+dtil))-1.0)
        return(
            mu/(lam+mu)*xi*q/(R*(R+dtil)**2),
            mu/(lam+mu)*(-np.sin(delta)/R+q*np.cos(delta)/R*Retaterm)-K3,
            K3
        )
    else:
        K3=mu/(lam+mu)/np.cos(delta)*(q/R*Retaterm-ytil/(R*(R+dtil)))
        return(
            mu/(lam+mu)*xi/np.cos(delta)*(1.0/(R*(R+dtil))-np.sin(delta)/R*Retaterm),
            mu/(lam+mu)*(-np.sin(delta)/R+q*np.cos(delta)/R*Retaterm)-K3,
            K3 
        )

Putting it all together:

In [20]:
def tilt_sim (stalon, stalat, evlon, evlat, evdep, strike, dip, rake, mu, lam, opening, U, L, W) :
    U1, U2, U3, x, y, d, mu, lam, delta, p, q, delta = coords_sim (stalon, stalat, evlon, evlat, evdep, strike, dip, rake, mu, lam, opening, U)
    xi, eta, ytil, dtil, R, X = chinnery_sim (x, p, q, L, W, delta)
    uzx, uzy = [], []
    for i in range(2):
        for j in range (2):
            K1, K2, K3 = tilt_K_sim (xi[i], eta[j], q, ytil[j], dtil[j], R[i][j], lam, mu, delta)
            uzx_strike, uzy_strike = tilt_strike_sim (xi[i], eta[j], q, ytil[j], dtil[j], R[i][j], K1, K2, delta, U1)
            uzx_dip, uzy_dip = tilt_dip_sim (xi[i], eta[j], q, ytil[j], dtil[j], R[i][j], K1, K3, delta, U2)
            uzx_tens, uzy_tens = tilt_tens_sim (xi[i], eta[j], q, ytil[j], dtil[j], R[i][j], K1, K3, delta, U3)
            uzx.append(uzx_strike + uzx_dip + uzx_tens)
            uzy.append(uzy_strike + uzy_dip + uzy_tens)
    return ( #chinnery
        uzx[0]-uzx[1]-uzx[2]+uzx[3], #uzx
        uzy[0]-uzy[1]-uzy[2]+uzy[3], #uzy
    )

Testing with Table 2 of Okada 1985:

In [21]:
stalon, stalat, stadep, evlon, evlat, evdep = -3.0, 2.0, 0.0, 0.0, 0.0, 4.0 #x=2, y=3, d=4
strike, dip = 0.0, 70.0 #degrees
mu, lam = 30, 30 #GPa
L, W = 3.0, 2.0

print("Strike-Slip Results: -5.175E-3, +2.945E-4")
rake, opening, U = 0.0, 0.0, 1.0
uzx, uzy = tilt_sim (stalon, stalat, evlon, evlat, evdep, strike, dip, rake, mu, lam, opening, U, L, W)
print("uzx={:0.3e}".format(uzx))
print("uzy={:0.3e}".format(uzy))

print("- - - - - - - - - - - - - - - - - - - - - - - - - -")

print("Dip-Slip Results: +4.088E-3, +2.626E-3")
rake, opening, U = 90.0, 0.0, 1.0
uzx, uzy = tilt_sim (stalon, stalat, evlon, evlat, evdep, strike, dip, rake, mu, lam, opening, U, L, W)
print("uzx={:0.3e}".format(uzx))
print("uzy={:0.3e}".format(uzy))

print("- - - - - - - - - - - - - - - - - - - - - - - - - -")

print("Tensile Results: -3.730E-4, +1.040E-2")
rake, opening, U = 0.0, 1.0, 0.0
uzx, uzy = tilt_sim (stalon, stalat, evlon, evlat, evdep, strike, dip, rake, mu, lam, opening, U, L, W)
print("uzx={:0.3e}".format(uzx))
print("uzy={:0.3e}".format(uzy))

Strike-Slip Results: -5.175E-3, +2.945E-4
uzx=-5.175e-03
uzy=2.945e-04
- - - - - - - - - - - - - - - - - - - - - - - - - -
Dip-Slip Results: +4.088E-3, +2.626E-3
uzx=4.088e-03
uzy=2.626e-03
- - - - - - - - - - - - - - - - - - - - - - - - - -
Tensile Results: -3.730E-4, +1.040E-2
uzx=-3.730e-04
uzy=1.040e-02


### **4. Refine the code to apply to real-life earthquakes from the Global CMT solution.**

In [22]:
import numpy as np

There are three major distinctions between the Okada model code simulation and the Okada model applied to real-life earthquakes.

1. The first is that the distance between the station and the event origin is more accurately determined through *obspy.gps2dist_azimuth* rather than simple subtraction. Once again, the Okada model uses a planar approximation, so larger distances will from the epicenter where the Earth's curvature is more pronounced, will not be as well-represented.

2. The second distinction is Chinnery Notation. To apply the Chinnery notation to the Global CMT, we need to adjust it so that the corners of the rupture are with respect to a centroid.

\begin{equation} f(\xi, \eta) \parallel = f(x+\frac{L}{2}, p+\frac{W}{2}) - f(x+\frac{L}{2}, p-\frac{W}{2}) - f(x-\frac{L}{2}, p+\frac{W}{2}) + f(x-\frac{L}{2}, p-\frac{W}{2}) \tag{20} \end{equation}

3. The third distinction is the removal of the tensile opening component. The Global CMT only considers shear dislocations, and thus have no opening component. Furthermore, since $U_3$ cannot be derived from the scalar moment, then $\Delta\Sigma$ must be measured separately, which is not possible from the CMT catalog.

### 4.1. Translation into Okada's Coordinate System

In [23]:
def coords (stalon, stalat, evlon, evlat, evdep, strike, dip, rake, mu, lam, opening, U):
    distance_m, azimuth, baz = gps2dist_azimuth(evlat, evlon, stalat, stalon)
    dEast = distance_m*np.sin(azimuth*np.pi/180.0) #m
    dNorth = distance_m*np.cos(azimuth*np.pi/180.0) #m

    x = dEast * np.sin(strike*np.pi/180.0) + dNorth * np.cos(strike*np.pi/180.0) #m
    y = -dEast * np.cos(strike*np.pi/180.0) + dNorth * np.sin(strike*np.pi/180.0) #m
    d = abs(evdep) #m
    delta = dip*np.pi/180.0 #radians
    
    p = y*np.cos(delta)+d*np.sin(delta) #m
    q = y*np.sin(delta)-d*np.cos(delta) #m

    mu = 10**9*mu #convert GPa to N/m^2
    lam = 10**9*lam #convert GPa to N/m^2
    return(
        U*np.cos(rake*np.pi/180.0), #m #U1
        U*np.sin(rake*np.pi/180.0), #m #U2
        x, #m #x
        y, #m #y
        d, #m #d
        mu, #GPa->N/m^2 #mu
        lam, #GPa->N/m^2 #lam
        delta, #rad #delta
        p, #m #p
        q, #m #q
        delta #radians #delta
    )

def chinnery (x, p, q, L, W, delta):
        xi = [x+L/2,x-L/2] #m #xi
        eta = [p+W/2,p-W/2] #m #eta

        ytil = [0,0]
        dtil = [0,0]
        R = [[0,0],[0,0]]
        X = [0,0]

        for i in range(2):
            for j in range(2):
                ytil[j]=eta[j]*np.cos(delta)+q*np.sin(delta) #m #ytil
                dtil[j]=eta[j]*np.sin(delta)-q*np.cos(delta) #m #dtil
                R[i][j]=np.sqrt(xi[i]**2.0+eta[j]**2.0+q**2.0) #m #R
                X[i]=np.sqrt(xi[i]**2+q**2) #m #X

        return(xi, eta, ytil, dtil, R, X)

### 4.2. Diplacements

In [24]:
def disp_strike (xi, eta, q, ytil, dtil, R, I1, I2, I4, delta, U1):
    if(q==0):
        atanterm=0
    else:
        atanterm=np.atan((xi*eta)/(q*R))
    if ((R+eta)==0):
        Retaterm=0
    else:
        Retaterm=1/(R+eta)
    return (
    -U1/(2.0*np.pi)*(((xi*q)/R*Retaterm)+atanterm+I1*np.sin(delta)),
    -U1/(2.0*np.pi)*(((ytil*q)/R*Retaterm)+(q*np.cos(delta))*Retaterm+I2*np.sin(delta)),
    -U1/(2.0*np.pi)*(((dtil*q)/R*Retaterm)+(q*np.sin(delta))*Retaterm+I4*np.sin(delta))
    )

def disp_dip (xi, eta, q, ytil, dtil, R, I1, I3, I5, delta, U2):
    if(q==0):
        atanterm=0
    else:
        atanterm=np.atan((xi*eta)/(q*R))
    return (
    -U2/(2.0*np.pi)*(((q/R)-I3*np.sin(delta)*np.cos(delta))),
    -U2/(2.0*np.pi)*(((ytil*q)/(R*(R+xi))+np.cos(delta)*atanterm-I1*np.sin(delta)*np.cos(delta))),
    -U2/(2.0*np.pi)*(((dtil*q)/(R*(R+xi))+np.sin(delta)*atanterm-I5*np.sin(delta)*np.cos(delta)))
    )

def disp_I (xi, eta, q, ytil, dtil, R, X, lam, mu, delta):
    if ((R+eta)==0):
        lnterm = -np.log(R-eta)
    else:
        lnterm = np.log(R+eta)
    if (np.cos(delta)==0):
        I3 = mu/(2*(lam+mu))*(eta/(R+dtil)+(ytil*q)/(R+dtil)**2-lnterm)
        I2_1stterm =mu/(lam+mu)*(-lnterm)-I3
        return(
            -mu/(2*(lam+mu))*xi*q/(R+dtil)**2,
            I2_1stterm,
            I3,
            mu/(lam+mu)*q/(R+dtil),
            mu/(lam+mu)*xi*np.sin(delta)/(R+dtil)
        )
    else:
        I4 = mu/(lam+mu)/np.cos(delta)*(np.log(R+dtil)-np.sin(delta)*lnterm)
        I3 = mu/(lam+mu)*(1/np.cos(delta)*ytil/(R+dtil)-lnterm)+np.sin(delta)/np.cos(delta)*I4
        I5 = 0 if xi==0 else mu/(lam+mu)*2.0/np.cos(delta)*np.atan((eta*(X+q*np.cos(delta))+X*(R+X)*np.sin(delta))/(xi*(R+X)*np.cos(delta)))
        I2_1stterm =mu/(lam+mu)*(-lnterm)-I3
        return(
            mu/(lam+mu)*(-1)/np.cos(delta)*xi/(R+dtil)-np.sin(delta)/(np.cos(delta))*I5,
            I2_1stterm,
            I3,
            I4, 
            I5
        )

All together:

In [25]:
def displacement (stalon, stalat, evlon, evlat, evdep, strike, dip, rake, mu, lam, opening, U, L, W) :
    U1, U2, x, y, d, mu, lam, delta, p, q, delta = coords (stalon, stalat, evlon, evlat, evdep, strike, dip, rake, mu, lam, opening, U)
    xi, eta, ytil, dtil, R, X = chinnery (x, p, q, L, W, delta)
    ux, uy, uz = [], [], []
    for i in range(2):
        for j in range (2):
            I1, I2, I3, I4, I5 = disp_I (xi[i], eta[j], q, ytil[j], dtil[j], R[i][j], X[i], lam, mu, delta)
            ux_strike, uy_strike, uz_strike = disp_strike (xi[i], eta[j], q, ytil[j], dtil[j], R[i][j], I1, I2, I4, delta, U1)
            ux_dip, uy_dip, uz_dip = disp_dip (xi[i], eta[j], q, ytil[j], dtil[j], R[i][j], I1, I3, I5, delta, U2)
            ux.append(ux_strike + ux_dip)
            uy.append(uy_strike + uy_dip)
            uz.append(uz_strike + uz_dip)
    return ( #chinnery
        ux[0]-ux[1]-ux[2]+ux[3], #ux
        uy[0]-uy[1]-uy[2]+uy[3], #uy
        uz[0]-uz[1]-uz[2]+uz[3] #uz
    )

### 4.3. Strains

In [31]:
def strain_strike (xi, eta, q, ytil, dtil, R, J1, J2, J4, delta, U1):
    Aeta=(2*R+eta)/(R**3*(R+eta)**2)
    if ((R+eta)==0):
        Retaterm=0
    else:
        Retaterm=1/(R+eta)
    return (
        U1/(2.0*np.pi)*(xi**2.0*q*Aeta-J1*np.sin(delta)),
        U1/(2.0*np.pi)*((xi**3.0*dtil)/(R**3.0*(eta**2.0+q**2.0))-(xi**3.0*Aeta+J2)*np.sin(delta)),
        U1/(2.0*np.pi)*(xi*q/R**3.0*np.cos(delta)+(xi*q**2.0*Aeta-J2)*np.sin(delta)),
        U1/(2.0*np.pi)*(ytil*q/R**3.0*np.cos(delta)+(q**3.0*Aeta*np.sin(delta)-2.0*q*np.sin(delta)/R*Retaterm-(xi**2+eta**2)/R**3*np.cos(delta)-J4)*np.sin(delta))
    )

def strain_dip (xi, eta, q, ytil, dtil, R, J1, J2, J3, delta, U2):
    Axi=(2*R+xi)/(R**3*(R+xi)**2)
    if ((R+eta)==0):
        Retaterm=0
    else:
        Retaterm=1/(R+eta)
    return (
        U2/(2.0*np.pi)*(xi*q/R**3.0+J3*np.sin(delta)*np.cos(delta)),
        U2/(2.0*np.pi)*(ytil*q/R**3.0-np.sin(delta)/R+J1*np.sin(delta)*np.cos(delta)),
        U2/(2.0*np.pi)*(ytil*q/R**3.0+q*np.cos(delta)/R*Retaterm+J1*np.sin(delta)*np.cos(delta)),
        U2/(2.0*np.pi)*(ytil**2*q*Axi-(2.0*ytil/(R*(R+xi))+xi*np.cos(delta)/R*Retaterm)*np.sin(delta)+J2*np.sin(delta)*np.cos(delta))
    )

def strain_J (xi, eta, q, ytil, dtil, R, lam, mu, delta):
    K1, K2, K3 = tilt_K (xi, eta, q, ytil, dtil, R, lam, mu, delta)
    if ((R+eta)==0):
        Retaterm=0
    else:
        Retaterm=1/(R+eta)
    if (np.cos(delta)==0):
        J1=mu/(2.0*(lam+mu))*q/(R+dtil)**2.0*(2.0*xi**2.0/(R*(R+dtil))-1.0)
        J2=mu/(2.0*(lam+mu))*xi*np.sin(delta)/(R+dtil)**2.0*(2.0*xi**2.0/(R*(R+dtil))-1)
        return(
            J1,
            J2,
            mu/(lam+mu)*(-xi/R*Retaterm)-J2,
            mu/(lam+mu)*(-np.cos(delta)/R-q*np.sin(delta)/R*Retaterm)-J1
        )
    else:
        J1=mu/(lam+mu)/np.cos(delta)*(xi**2.0/(R*(R+dtil)**2.0)-1.0/(R+dtil))-np.sin(delta)/np.cos(delta)*K3
        J2=mu/(lam+mu)/np.cos(delta)*xi*ytil/(R*(R+dtil)**2.0)-np.sin(delta)/np.cos(delta)*K1
        return(
            J1,
            J2,
            mu/(lam+mu)*(-xi/R*Retaterm)-J2,
            mu/(lam+mu)*(-np.cos(delta)/R-q*np.sin(delta)/R*Retaterm)-J1
        )

All together:

In [56]:
def strain (stalon, stalat, evlon, evlat, evdep, strike, dip, rake, mu, lam, opening, U, L, W) :
    U1, U2, x, y, d, mu, lam, delta, p, q, delta = coords (stalon, stalat, evlon, evlat, evdep, strike, dip, rake, mu, lam, opening, U)
    xi, eta, ytil, dtil, R, X = chinnery (x, p, q, L, W, delta)
    uxx, uxy, uyx, uyy = [], [], [], []
    for i in range(2):
        for j in range (2):
            J1, J2, J3, J4 = strain_J (xi[i], eta[j], q, ytil[j], dtil[j], R[i][j], lam, mu, delta)
            uxx_strike, uxy_strike, uyx_strike, uyy_strike = strain_strike (xi[i], eta[j], q, ytil[j], dtil[j], R[i][j], J1, J2, J4, delta, U1)
            uxx_dip, uxy_dip, uyx_dip, uyy_dip = strain_dip (xi[i], eta[j], q, ytil[j], dtil[j], R[i][j], J1, J2, J3, delta, U2)
            uxx.append(uxx_strike + uxx_dip)
            uxy.append(uxy_strike + uxy_dip)
            uyx.append(uyx_strike + uyx_dip)
            uyy.append(uyy_strike + uyy_dip)
    return ( #chinnery
        uxx[0]-uxx[1]-uxx[2]+uxx[3], #uxx
        uxy[0]-uxy[1]-uxy[2]+uxy[3], #uxy
        uyx[0]-uyx[1]-uyx[2]+uyx[3], #uyx
        uyy[0]-uyy[1]-uyy[2]+uyy[3] #uyy
    )

### 4.4. Tilts

In [33]:
def tilt_strike (xi, eta, q, ytil, dtil, R, K1, K2, delta, U1):
    Aeta=(2*R+eta)/(R**3*(R+eta)**2)
    return (
        U1/(2.0*np.pi)*(-xi*q**2.0*Aeta*np.cos(delta)+(xi*q/R**3.0-K1)*np.sin(delta)),
        U1/(2.0*np.pi)*(dtil*q/R**3*np.cos(delta)+(xi**2.0*q*Aeta*np.cos(delta)-np.sin(delta)/R+ytil*q/R**3-K2)*np.sin(delta))
    )

def tilt_dip (xi, eta, q, ytil, dtil, R, K1, K3, delta, U2):
    Axi=(2*R+xi)/(R**3*(R+xi)**2)
    if ((R+eta)==0):
        Retaterm=0
    else:
        Retaterm=1/(R+eta)
    return (
        U2/(2.0*np.pi)*(dtil*q/R**3+q*np.sin(delta)/R*Retaterm+K3*np.sin(delta)*np.cos(delta)),
        U2/(2.0*np.pi)*(ytil*dtil*q*Axi-(2.0*dtil/(R*(R+xi))+xi*np.sin(delta)/R*Retaterm)*np.sin(delta)+K1*np.sin(delta)*np.cos(delta))
    )

def tilt_K (xi, eta, q, ytil, dtil, R, lam, mu, delta):
    if ((R+eta)==0):
        Retaterm=0
    else:
        Retaterm=1/(R+eta)
    if (np.cos(delta)==0):
        K3=mu/(lam+mu)*np.sin(delta)/(R+dtil)*(xi**2/(R*(R+dtil))-1.0)
        return(
            mu/(lam+mu)*xi*q/(R*(R+dtil)**2),
            mu/(lam+mu)*(-np.sin(delta)/R+q*np.cos(delta)/R*Retaterm)-K3,
            K3
        )
    else:
        K3=mu/(lam+mu)/np.cos(delta)*(q/R*Retaterm-ytil/(R*(R+dtil)))
        return(
            mu/(lam+mu)*xi/np.cos(delta)*(1.0/(R*(R+dtil))-np.sin(delta)/R*Retaterm),
            mu/(lam+mu)*(-np.sin(delta)/R+q*np.cos(delta)/R*Retaterm)-K3,
            K3 
        )

All together:

In [34]:
def tilt (stalon, stalat, evlon, evlat, evdep, strike, dip, rake, mu, lam, opening, U, L, W) :
    U1, U2, x, y, d, mu, lam, delta, p, q, delta = coords (stalon, stalat, evlon, evlat, evdep, strike, dip, rake, mu, lam, opening, U)
    xi, eta, ytil, dtil, R, X = chinnery (x, p, q, L, W, delta)
    uzx, uzy = [], []
    for i in range(2):
        for j in range (2):
            K1, K2, K3 = tilt_K (xi[i], eta[j], q, ytil[j], dtil[j], R[i][j], lam, mu, delta)
            uzx_strike, uzy_strike = tilt_strike (xi[i], eta[j], q, ytil[j], dtil[j], R[i][j], K1, K2, delta, U1)
            uzx_dip, uzy_dip = tilt_dip (xi[i], eta[j], q, ytil[j], dtil[j], R[i][j], K1, K3, delta, U2)
            uzx.append(uzx_strike + uzx_dip)
            uzy.append(uzy_strike + uzy_dip)
    return ( #chinnery
        uzx[0]-uzx[1]-uzx[2]+uzx[3], #uzx
        uzy[0]-uzy[1]-uzy[2]+uzy[3], #uzy
    )

### **5. Compute the surface deformations at all relevant NGL stations from the 10 largest earthquakes in Greece.**

### 5.1 Rotating results back into North, East, and Up coordinate system [mm] to follow IGS Workflow.

In [35]:
def NE_rotate(ux, uy, uz, strike):
    return(
        (ux * np.cos(strike*np.pi/180.0) + uy * np.sin(strike*np.pi/180.0))*10.0**3.0, #mm #dN
        (ux * np.sin(strike*np.pi/180.0) - uy * np.cos(strike*np.pi/180.0))*10.0**3.0, #mm #dE
        uz*10.0**3.0 #mm #dH
    )

First we load the earthquakes:

In [36]:
from obspy import read_events
from obspy import UTCDateTime
from obspy.geodetics import gps2dist_azimuth

In [37]:
big10_catalog = read_events("Big10_Greece_Seismicity.xml")
print(len(big10_catalog))

10


Then we load the stations:

In [38]:
stations = pd.read_csv("NGL_Greece_Stations.csv", sep=",")
stations_ids = stations["Station_ID"].astype(str)
stations_lats = pd.to_numeric(stations["Latitude"], errors="coerce")
stations_lons = pd.to_numeric(stations["Longitude"], errors="coerce")
print(len(stations))

270


### 5.2. Applying Scaling Relations to know *U, L,* and *W*

The IGS Network follows the following convention for scaling laws, based on Métivier et al.'s work [9].

For earthquakes with magnitudes $<7.6$, the width, length, and slip of the rupture are estimated from Yen & Ma [10]. Thus, for earthquakes with magnitude $<7.27$ or scalar moment $\leq 10^{20} [N-m]$:

\begin{cases}
\log L = \frac{1}{2}\log M_0-8.08 \\
\log W = \frac{1}{2}\log M_0-8.08\\
\log U = 1.68\pm0.33
\tag{21}
\end{cases}

On the other hand, for earthquakes with magnitude $<7.27$ or scalar moment $\leq 10^{20} [N-m]$:

\begin{cases}
\log L = \frac{1}{3}\log M_0-4.84 \\
\log W = \frac{1}{3}\log M_0-5.27 \\
\log U = \frac{1}{3}\log M_0-4.37
\tag{22}
\end{cases}

For earthquakes with magnitudes $\geq7.6$, the width, length, and slip of the rupture are estimated from Mai & Beroza [11].

\begin{cases}
\log L = 0.39\log M_0-6.13 \\
\log W = 0.32\log M_0-5.05 \\
\log U = 0.29\log M_0-3.34
\tag{23}
\end{cases}

***

[9] L. Métivier, X. Collilieux, D. Lercier, Z. Altamimi, and F. Beauducel, “Global coseismic deformations, GNSS time series analysis, and Earthquake Scaling Laws,” Journal of Geophysical Research: Solid Earth, vol. 119, no. 12, pp. 9095–9109, Dec. 2014. doi:10.1002/2014jb011280

[10] Y.-T. Yen and K.-F. Ma, “Source-scaling relationship for M 4.6-8.9 earthquakes, specifically for earthquakes in the collision zone of Taiwan,” Bulletin of the Seismological Society of America, vol. 101, no. 2, pp. 464–481, Mar. 2011. doi:10.1785/0120100046

[11] P. M. Mai and G. C. Beroza, “Source scaling properties from finite-fault-rupture models,” Bulletin of the Seismological Society of America, vol. 90, no. 3, pp. 604–615, Jun. 2000. doi:10.1785/0119990126 

In [48]:
def scaling (mag, m0):
    if (mag<7.6): #Yen & Ma
        if (m0<=10.0**20.0): #N-m
            return(
                10.0**(0.5*np.log10(m0)-8.08)*1000.0, #m #L
                10.0**(0.5*np.log10(m0)-8.08)*1000.0, #m #W
                10.0**1.68/100 #m #U
            )
        else:
            return(
                10.0**(1.0/3.0*np.log10(m0)-4.84)*1000.0, #m #L
                10.0**(1.0/3.0*np.log10(m0)-5.27)*1000.0, #m #W
                10.0**(1.0/3.0*np.log10(m0)-4.37)/100 #m #U
            )
    else: #Mai & Beroza
        return(
            10.0**(0.39*np.log10(m0)-6.13)*1000.0, #m #L
            10.0**(0.32*np.log10(m0)-5.05)*1000.0, #m #W
            10.0**(0.29*np.log10(m0)-3.34)/100 #m #U
        )

### 5.3. Computing Surface Deformations

In [57]:
j=1
for event in big10_catalog: # For each earthquake

    # Read earthquake data
    origin = event.preferred_origin() or event.origins[0]
    event_time = origin.time
    focal_mech = event.preferred_focal_mechanism() or event.focal_mechanisms[0]
    moment_tensor = focal_mech.moment_tensor
    nodal_planes = focal_mech.nodal_planes
    plane1 = nodal_planes.nodal_plane_1 # Since it's a point source, either is ok

    # Declare variables
    evlat = origin.latitude #degN
    evlon = origin.longitude #degE
    evdep = origin.depth #obspy automatically converts km to m
    
    strike = plane1.strike #deg
    dip = plane1.dip #deg
    rake = plane1.rake #deg
    
    mag = event.preferred_magnitude() or event.magnitudes[0]
    mag = float(mag.mag)
    m0 = moment_tensor.scalar_moment #obspy automatically converts from dyne-cm to N-m
    
    mu = 30.0 #GPa #mu
    lam = 30.0 #GPa #lambda

    # Station Parameter Lists
    sta_ids = [] #station IDs
    sta_lats = [] #stalat list
    sta_lons = [] #stalon list

    # Okada 1985 Point Source Lists
    # Displacements in meters
    sta_ux = []
    sta_uy = []
    sta_uz = []
    sta_tot = [] #total displacement

    # Strains
    sta_uxx = []
    sta_uxy = []
    sta_uyx = []
    sta_uyy = []

    # Tilts
    sta_uzx = []
    sta_uzy = []

    # Displacements in IGS conventions (milimeters)
    sta_dE = []
    sta_dN = []
    sta_dH = []
    
    relevant_stations = pd.DataFrame()
    
    for i in range(len(stations_ids)): # For every station

        #Compute the slip (U), length (L), and width of rupture (W) using scaling laws
        L, W, U = scaling(mag, m0)
        
        # Compute the Total Displacement
        ux, uy, uz = displacement (stations_lons[i], stations_lats[i], evlon, evlat, evdep, strike, dip, rake, mu, lam, opening, U, L, W)
        total_disp = np.sqrt(ux**2+uy**2+uz**2)

        if (total_disp>=10**(-3)): #If the Total Displacement is larger than 1 mm, compute Full Deformation Profile
            uxx, uxy, uyx, uyy = strain (stations_lons[i], stations_lats[i], evlon, evlat, evdep, strike, dip, rake, mu, lam, opening, U, L, W)
            uzx, uzy = tilt (stations_lons[i], stations_lats[i], evlon, evlat, evdep, strike, dip, rake, mu, lam, opening, U, L, W)
            dN, dE, dH = NE_rotate(ux, uy, uz, strike)

            # Save values
            # Displacement
            sta_ux.append(ux)
            sta_uy.append(uy)
            sta_uz.append(uz)
            sta_tot.append(total_disp*10.0**3.0) #mm
        
            # Strains
            sta_uxx.append(uxx)
            sta_uxy.append(uxy)
            sta_uyx.append(uyx)
            sta_uyy.append(uyy)
        
            # Tilts
            sta_uzx.append(uzx)
            sta_uzy.append(uzy)

            # IGS conventions
            sta_dE.append(dE)
            sta_dN.append(dN)
            sta_dH.append(dH)

            #stations
            sta_ids.append(stations_ids[i])
            sta_lats.append(stations_lats[i])
            sta_lons.append(stations_lons[i])
        else:
            continue

    # Save in dataframe
    relevant_stations["Station_ID"] = sta_ids
    relevant_stations["Latitude"] = sta_lats
    relevant_stations["Longitude"] = sta_lons

    relevant_stations["Total Displacement [mm]"] = sta_tot

    relevant_stations["dE [mm]"] = sta_dE
    relevant_stations["dN [mm]"] = sta_dN
    relevant_stations["dH [mm]"] = sta_dH
    
    relevant_stations["ux [m]"] = sta_ux
    relevant_stations["uy [m]"] = sta_uy
    relevant_stations["uz [m]"] = sta_uz
    
    relevant_stations["uxx"] = sta_uxx
    relevant_stations["uxy"] = sta_uxy
    relevant_stations["uyx"] = sta_uyx
    relevant_stations["uyy"] = sta_uyy
    
    relevant_stations["uzx"] = sta_uzx
    relevant_stations["uzy"] = sta_uzy
    
    relevant_stations.to_csv(f"Finite/Greece_EQ{j}_M{mag}_{evlon}dE_{evlat}dN_{evdep}[m]dH_{event_time}.csv", index=False)
    j=j+1